# UMAP and cell states

**UMAP** (Uniform Manifold Approximation and Projection) draws every cell as a point on a 2D map so that **cells with similar foci phenotypes lie close together**.
It doesn't summarise variation along straight axes like PCA. It preserves **neighbourhoods**, so it can reveal **groups of cells**, i.e. *cell states*, that PCA blurs together: for example "cells with a few large round foci" next to "cells with many small irregular foci".

This notebook answers:
* **Are there distinct cell states**, or do cells form a continuum?
* **What characterises each state**, in terms of foci metrics and example images?
* **Do conditions differ in how their cells are distributed over the states?** For example, *"the mutant has 3× more cells in the many-small-foci state."*

### How it relates to the other notebooks

| Notebook | Question |
|---|---|
| `METRICS.ipynb` | which single metric changes, and by how much (**main result**) |
| `PCA_v2.ipynb` | whether the overall phenotype shifts, and along which combination of metrics |
| `CSAT.ipynb` | whether the saturation concentration changes |
| **`UMAP_v2.ipynb`** | whether there are **sub-populations of cells**, and whether conditions change their **proportions** |

### Rules for reading a UMAP

UMAP is a map for **looking**, not for **measuring**:
* **Distances between clusters, cluster sizes and densities on the map are not meaningful.** Two groups that look far apart may be similar, and a tight blob may be very variable.
* **The axes have no meaning,** and the map can be rotated or mirrored. That's why there are no axis values.
* **The picture depends on settings** (`n_neighbors`, `min_dist`) and on the random seed. Section 4 checks that the structures you interpret appear under all reasonable settings.

So in this notebook:
1. **Clusters are found in the original feature space** (all z-scored metrics), not on the UMAP, and are only *drawn* on it. They don't change when the UMAP settings change.
2. **Nothing is tested on UMAP coordinates.** Tests compare the **share of each condition's cells in each cluster**, per replicate.

### Workflow

| Step | Section |
|---|---|
| 1 | Parameters |
| 2–3 | Load, design check, feature preparation (as in `PCA_v2.ipynb`) |
| 4 | Choosing the UMAP settings: sweep, trustworthiness, seed stability |
| 5 | Conditions on the map |
| 6 | Reading the map: each feature as a colour |
| 7 | Cell states: how many, what they are, example cells |
| 8 | Cluster composition of conditions: replicate-level tests |
| 9 | Save and report |

In [ ]:
%matplotlib inline
from UMAP_v2_utils import *

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 160)

# 1. Parameters

`data`, the features, `LOG_FEATURES`, `MISSING` and `MIN_FOCI` work exactly as in `PCA_v2.ipynb`; see its annotations. In short:
* include foci properties, not intensities or expression;
* avoid near-duplicate features;
* log-transform skewed features.

### UMAP settings

* **`N_NEIGHBORS`**: how many neighbours each cell "looks at". **Small** (5–10) emphasises fine local structure and splits the map into many islands, some of them noise. **Large** (30–50) shows the global layout and merges small groups.
  15–30 is a good default, and it must be well below the number of cells.
* **`MIN_DIST`**: how tightly points may be packed. **Small** (0.0–0.1) makes compact clumps, which is good for seeing clusters. **Large** (0.5) spreads points evenly, which is good for seeing gradients.
* **`SEED`**: UMAP is random. Fix the seed for a reproducible figure, and check other seeds in section 4.

### Clusters

* **`N_CLUSTERS = "auto"`** chooses k from silhouette and bootstrap stability (section 7). You can also set a number, e.g. after looking at section 7, but justify it.

In [ ]:
# Root directory where the data is stored
root_dir = "/mnt/c/Users/helen/Desktop/data"

# One entry per condition x replicate (same format as METRICS.ipynb / PCA_v2.ipynb)
data = [
    {"condition": "WT",   "replicate": "1", "path": f"{root_dir}/WT/gmm/aggregated_data.csv"},
    {"condition": "MGS3", "replicate": "1", "path": f"{root_dir}/MGS3/gmm/aggregated_data.csv"},
]

COMPARISONS = "vs_reference"   # "vs_reference" or "all_pairs"
REFERENCE = "WT"
EXCLUDE = []                   # cells to leave out: (condition, replicate, image_id)

features = [
    # Amount of phase separation
    "n_foci", "foci_density", "area_fraction", "fraction_signal_in_foci", "partition_coefficient",
    # Focus properties
    "area", "eccentricity", "solidity", "size_gini",
    # Spatial organisation
    "clark_evans_ratio", "nn_gap", "fraction_touching",
]
LOG_FEATURES = ["n_foci", "foci_density", "area_fraction", "partition_coefficient", "area", "nn_gap"]
MISSING = "drop"               # "drop" or "median"
MIN_FOCI = {"size_gini": 4, "clark_evans_ratio": 4, "nn_gap": 2, "fraction_touching": 2}

# UMAP
N_NEIGHBORS = 15
MIN_DIST = 0.3
SEED = 42

# Clusters (cell states), found in feature space
N_CLUSTERS = "auto"            # "auto" or a number
K_RANGE = range(2, 9)          # k values tried by "auto"

ALPHA = 0.05
SAVE_RESULTS = True
output_dir = Path(root_dir) / "umap_v2"   # tables and figures are saved here

# 2–3. Load the data and prepare the features

This is the same preparation as `PCA_v2.ipynb`: the log transform, removal (or filling) of cells with missing features, and z-scoring. `kept_%` is the share of each condition's cells that enter the analysis.

**Number of cells:** UMAP and clustering need many cells. With fewer than ~100 cells per condition, the map is dominated by a few points and clusters are unstable (section 7 measures this).

In [ ]:
df = load_conditions(data)
if EXCLUDE:
    key = list(zip(df["condition"].astype(str), df["replicate"].astype(str), df["image_id"].astype(str)))
    df = df[[k not in set(map(tuple, EXCLUDE)) for k in key]]
df = apply_min_foci(df, MIN_FOCI)
TEST_LEVEL = check_design(df)
palette = condition_palette(list(df["condition"].cat.categories))
display(design_table(df))

data_u, Z, kept = prepare_features(df, features, log_features=LOG_FEATURES, missing=MISSING)
kept

# 4. Choosing the UMAP settings

### Sweep over `n_neighbors` and `min_dist`

The same cells are shown with different settings and coloured by condition. Each panel gives its **trustworthiness**, the share of each cell's nearest neighbours on the map that are also neighbours in the full feature space:
**> 0.9 good, 0.8–0.9 acceptable, < 0.8 the map distorts neighbourhoods.**

**Interpret only structures that appear in most panels.** An island that exists only at `n_neighbors = 5` is probably noise. Choose settings with high trustworthiness that show the stable structure clearly, and set them in the Parameters.

In [ ]:
fig, sweep = plot_umap_sweep(Z, data_u["condition"].astype(str), palette)
if SAVE_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "umap_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
sweep.pivot(index="min_dist", columns="n_neighbors", values="trustworthiness")

### Seed stability

These are the chosen settings with three different random seeds. The maps may be rotated or mirrored, which is irrelevant, but **the same groups and gradients should appear in each**.

In [ ]:
fig = plot_seed_stability(Z, data_u["condition"].astype(str), palette, N_NEIGHBORS, MIN_DIST)
plt.show()

# 5. Conditions on the map

* **Dots:** cells, coloured by condition.
* **Contours:** the region holding the densest 50 % of each condition's cells (shown with ≥ 15 cells).

What to look for:
* **Shifted contours**: a condition's cells sit in a different part of the phenotype space;
* **an extra region** occupied by only one condition: a cell state that appears or disappears;
* **completely overlapping contours**: no visible difference. This is fine and a result in itself.

Remember that the **size** of the separation on the map isn't a measure of effect size. `PCA_v2.ipynb` section 8 gives that.

In [ ]:
emb = run_umap(Z, N_NEIGHBORS, MIN_DIST, SEED)
umap_df = add_embedding(data_u, emb)
print(f"Trustworthiness of this map: {trust(Z, emb):.3f}")

fig = plot_umap(umap_df, palette)
if SAVE_RESULTS:
    fig.savefig(output_dir / "umap_conditions.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_umap_facets(umap_df, palette)
if SAVE_RESULTS:
    fig.savefig(output_dir / "umap_facets.png", dpi=300, bbox_inches="tight")
plt.show()

# 6. Reading the map: each feature as a colour

UMAP axes have no meaning, so the map is interpreted by **colouring cells by each feature**. Red is above the average of all cells, blue below, in z-scores clipped at ±2.5 SD.
A feature that changes smoothly across the map is one of the main axes of variation; a feature that marks one region characterises that region.

The last panel shows **expression level** (`nucleus_mean_intensity`, log scale). If a region of the map is simply the high-expressing cells, its "cell state" may only reflect protein amount (see `CSAT.ipynb` and `METRICS.ipynb` section 7).

In [ ]:
extra = ["nucleus_mean_intensity"] if "nucleus_mean_intensity" in umap_df.columns else []
fig = plot_feature_overlays(umap_df, Z, extra=extra)
if SAVE_RESULTS:
    fig.savefig(output_dir / "umap_features.png", dpi=200, bbox_inches="tight")
plt.show()

# 7. Cell states

### How many?

Cells are clustered with **KMeans in the full z-scored feature space** for k = 2…8. Each k is scored by:
* **silhouette** (−1…1): how well separated the clusters are. **> 0.5** clear clusters, **0.25–0.5** some structure, **< 0.25** essentially no separation;
* **stability** (adjusted Rand index between the clustering of all cells and of 30 bootstrap resamples): **≥ 0.75** (dashed line) means the same clusters come back when the data are resampled.

With `N_CLUSTERS = "auto"`, k is the best-separated **stable** clustering without tiny clusters (< 5 % of cells). The message below says how much to trust it.
**If no k is stable, the cells form a continuum.** Clusters then only cut it into arbitrary pieces: use them for orientation, and describe the result with `PCA_v2.ipynb` instead.

In [ ]:
quality = choose_k(Z, K_RANGE)
k_auto, message = suggest_k(quality)
K = k_auto if N_CLUSTERS == "auto" else int(N_CLUSTERS)
print(message)
if K != k_auto:
    print(f"Using k = {K} as set in N_CLUSTERS.")

fig = plot_k_quality(quality, K)
if SAVE_RESULTS:
    fig.savefig(output_dir / "cluster_number.png", dpi=300, bbox_inches="tight")
plt.show()
quality.round(3)

### Clusters on the map

These are the clusters found in feature space, drawn on the UMAP and numbered by size (1 = largest).
Clusters that occupy **separate regions** of the map agree with UMAP's view of the data. A cluster **scattered over the map** is defined by a feature combination the 2D map doesn't show well. It isn't wrong, but read it from the profiles below.

In [ ]:
clusters = fit_clusters(Z, K)
cpal = cluster_palette(K)
umap_df["cluster"] = clusters

fig = plot_clusters_on_umap(umap_df, clusters, cpal)
if SAVE_RESULTS:
    fig.savefig(output_dir / "umap_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

### What characterises each cluster

Each cell of the heatmap is the **median z-score of a feature in a cluster**: **+1** means the cluster's typical cell is 1 SD above the average of all cells, **−1** 1 SD below.
Features are sorted by how strongly they distinguish any cluster. The table gives a plain-language summary (|z| ≥ 0.5).

To **name** a cluster, combine its profile with its example cells below, e.g. *"cluster 2: many small foci, low enrichment"*.

In [ ]:
profiles = cluster_profiles(Z, clusters)
fig = plot_cluster_profiles(profiles, cpal)
if SAVE_RESULTS:
    fig.savefig(output_dir / "cluster_profiles.png", dpi=300, bbox_inches="tight")
plt.show()
describe_clusters(profiles)

### Example cells: look at them

These are the five cells **closest to each cluster's centre**, its most typical members. Open their images and `panels/` overlays (`image_id`).
A cluster is only a meaningful cell state if its typical cells **look alike and differ visibly from the other clusters**. A cluster of segmentation failures, such as cut-off nuclei or noise called foci, should be removed with `EXCLUDE`, not reported.

In [ ]:
examples = typical_cells(umap_df, Z, clusters)
examples

# 8. Do conditions differ in their cell states?

For each cluster: **what percentage of a condition's cells fall into it**, compared with the reference.

* **Left:** cluster shares per condition, as the mean of replicates.
* **Right:** one panel per cluster, with each replicate's share (dots) and their mean (bar). Stars are q-values vs. the baseline.

| Column | Meaning |
|---|---|
| `pct`, `pct_baseline` | % of the condition's / baseline's cells in the cluster (all cells pooled) |
| `difference_pct` | difference in percentage points |
| `p_replicates`, `q_replicates` | Welch *t*-test on **replicate shares** (*n* = replicates). **Base conclusions on this one** |
| `p_cells`, `q_cells` | Fisher's exact test on cell counts. **Descriptive**: treats cells as independent |

The overall chi-square test (conditions × clusters, cell level) asks whether the composition differs at all. It's descriptive too, for the same reason.

**If the clusters are not stable (section 7), don't interpret these tests:** the cluster boundaries are arbitrary, so the shares are too.

In [ ]:
pairs = comparison_pairs(umap_df["condition"].cat.categories, reference=REFERENCE, mode=COMPARISONS)
comp_res, overall = composition_tests(umap_df, clusters, pairs)
q_col = "q_replicates" if TEST_LEVEL == "replicates" else "q_cells"
print(f"Overall chi-square (cells): chi2 = {overall['chi2']:.2f}, dof = {overall['dof']}, p = {overall['p']:.3g}")
print(f"Stars from: {q_col}")

fig = plot_composition(umap_df, clusters, cpal, palette, results=comp_res, q_col=q_col)
if SAVE_RESULTS:
    fig.savefig(output_dir / "cluster_composition.png", dpi=300, bbox_inches="tight")
plt.show()

comp_res[["cluster", "condition", "baseline", "n_rep", "pct", "pct_baseline", "difference_pct",
          "p_replicates", "q_replicates", "p_cells", "q_cells"]].round(4)

# 9. Save and report

In [ ]:
if SAVE_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    umap_df.to_csv(output_dir / "umap_cells.csv", index=False)
    profiles.to_csv(output_dir / "cluster_profiles.csv")
    quality.to_csv(output_dir / "cluster_number.csv", index=False)
    examples.to_csv(output_dir / "cluster_examples.csv", index=False)
    comp_res.to_csv(output_dir / "cluster_composition_tests.csv", index=False)
    composition(umap_df, clusters).to_csv(output_dir / "cluster_composition.csv", index=False)
    sweep.to_csv(output_dir / "umap_sweep.csv", index=False)
    print(f"Saved to {output_dir}")

### Reporting checklist

* **The features and preparation:** as in `PCA_v2.ipynb`: the feature list, log transform, z-scoring, and the % of cells kept.
* **The UMAP:** *"UMAP (n_neighbors = 15, min_dist = 0.3, Euclidean, on z-scored features; trustworthiness 0.9); structures were consistent across settings and seeds"* (section 4).
* **The clusters:** *"KMeans in feature space, k = 3 chosen by silhouette (0.31) and bootstrap stability (ARI 0.82)"*, a description of each cluster from its profile and example images, and the statement that the UMAP is used for display only.
* **The composition test:** *"% of cells per cluster compared between conditions by Welch's t-test on experiment-level shares, BH-corrected; n = 4 experiments"*.
* **If there are no stable clusters,** say so: *"cells formed a continuum without distinct states"*. This is a valid and common result.

Example sentence:
> *UMAP of 12 foci features revealed three reproducible cell states (KMeans in feature space, k = 3; silhouette 0.31; bootstrap ARI 0.82): cells with many small foci (cluster 1), few large round foci (cluster 2) and irregular, clustered foci (cluster 3). MGS3 shifted cells from cluster 2 to cluster 1 (41 % vs. 18 % of cells in cluster 1; q = 0.02, n = 4 experiments), without changing the share of cluster 3.*